# 04 · Formulação exata e experimentos com Gurobi

**Questão.** Qual é o menor número de escolas novas hipotéticas necessário para absorver a demanda, e como muda a distância total se permitirmos mais locais novos?

Modelo de **localização-alocação capacitada com rede existente fixa**. Comparamos \(p_{min}\), \(p_{min}+1\), \(p_{min}+2\) candidatos abertos, quando possível. A etapa 1 minimiza a contagem de aberturas; a etapa 2 minimiza quilômetros-estudante, impondo um limite no número de aberturas. Não há custos reais de construção neste exercício.

**Cuidado com o significado:** todos os candidatos são centrados em setores e não terrenos validados. As distâncias são em linha reta. Capacidade de novas escolas e participação escolar pública são hipóteses. Resultados não devem ser apresentados como decisões reais de infraestrutura.

## 1. Formulação matemática

Conjuntos: \(I\) setores de demanda; \(J\) escolas públicas existentes; \(K\) locais candidatos de novas escolas.

Parâmetros: \(w_i\) demanda estimada; \(q_j\) capacidade proxy existente; \(u_k\) capacidade hipotética nova; \(d_{ih}\) distância entre setor e escola existente ou candidata.

Variáveis: \(y_k\in\{0,1\}\) abertura de escola nova e \(z_{ih}\ge0\) fluxo de estudantes.

\[
\sum_{h\in J\cup K}z_{ih}=w_i \quad \forall i\in I
\]
\[
\sum_{i\in I}z_{ij}\le q_j \quad \forall j\in J
\]
\[
\sum_{i\in I}z_{ik}\le u_k y_k \quad \forall k\in K
\]

**Problema A:** minimizar \(\sum_k y_k\). **Problema B:** minimizar \(\sum_{i,h}d_{ih}z_{ih}\) sujeito a \(\sum_k y_k\le p\).

Usamos \(z\) contínuo porque, para \(y\) inteiro fixo, a subestrutura de transporte com demandas e capacidades inteiras admite uma solução básica ótima integral; o solucionador pode produzir pequenas tolerâncias numéricas. Validamos a quase integralidade dos fluxos e não fazemos arredondamento silencioso. A integralidade é uma propriedade da estrutura, não uma restrição implementada para \(z\).

Quando `DISTANCIA_MAXIMA_KM` é definido, pares mais distantes são removidos. A garantia de capacidade agregada \(\sum q + \sum u\ge \sum w\) **não assegura cobertura local** sob limite de distância.

## 2. Configuração, importação e transparência

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

raiz = Path.cwd().resolve()
if not (raiz / "pyproject.toml").exists():
    raiz = next((p for p in raiz.parents if (p / "pyproject.toml").exists()), raiz)
assert (raiz / "pyproject.toml").exists(), "Execute dentro do repositório PM."
saida = raiz / "outputs" / "localizacao_escolar" / "lavras"
saida.mkdir(parents=True, exist_ok=True)
CODIGO_MUNICIPIO = "3138203"
ETAPAS = {"fundamental_iniciais": ("V01032", "QT_MAT_FUND_AI", "QT_TUR_FUND_AI"),
          "fundamental_finais": ("V01033", "QT_MAT_FUND_AF", "QT_TUR_FUND_AF"),
          "medio": ("V01034", "QT_MAT_MED_PROP", "QT_TUR_MED_PROP")}
import gurobipy as gp
from gurobipy import GRB
from time import perf_counter
# Simulação: uma escola nova pode atender 12 turmas hipotéticas.
LIMIT_TIME_S = 120
THREADS = 2
MIP_GAP = 0.0
DISTANCIA_MAXIMA_KM = None   # None: sem limite operacional. Testar 1.5, 3.0, etc. separadamente.
CENARIOS = [("central", "central")]  # (demanda, capacidade); ampliar com ("alta", "conservadora")
ETAPAS_EXECUTADAS = list(ETAPAS)
VARIACOES_P = 2
config = {"time_limit_s": LIMIT_TIME_S, "threads": THREADS, "mip_gap": MIP_GAP,
          "distancia_maxima_km": DISTANCIA_MAXIMA_KM, "cenarios": CENARIOS,
          "etapas": ETAPAS_EXECUTADAS, "variacoes_p": VARIACOES_P,
          "gurobi_version": ".".join(map(str, gp.gurobi.version()))}
print(json.dumps(config, ensure_ascii=False, indent=2))

In [ ]:
def ler_instancia(etapa, cenario_demanda="central", cenario_capacidade="central"):
    pasta = saida / "instancias" / etapa
    d = pd.read_csv(pasta / "demanda.csv", dtype={"id": "string", "CD_SETOR": "string"}).set_index("id")
    e = pd.read_csv(pasta / "existentes.csv", dtype={"id": "string", "CO_ENTIDADE": "string"}).set_index("id")
    k = pd.read_csv(pasta / "candidatos.csv", dtype={"id": "string", "CD_SETOR": "string"}).set_index("id")
    dist = pd.read_csv(pasta / "distancias_km.csv", dtype={"id": "string"}).set_index("id")
    w = d["w_" + cenario_demanda].astype(int)
    q = e["q_" + cenario_capacidade].astype(int)
    u = k["u_" + cenario_capacidade].astype(int)
    assert w.ge(0).all() and q.ge(0).all() and u.gt(0).all()
    assert w.index.is_unique and q.index.is_unique and u.index.is_unique
    assert set(q.index).isdisjoint(set(u.index))
    dist = dist.reindex(index=w.index, columns=[*q.index, *u.index]).astype(float)
    assert np.isfinite(dist.to_numpy()).all() and dist.ge(0).all().all()
    return w, q, u, dist

def construir_modelo(w, q, u, dist, p=None, distancia_maxima=None, objetivo="distancia"):
    model = gp.Model("localizacao_alocacao_escolar")
    model.Params.OutputFlag = 0
    model.Params.TimeLimit = LIMIT_TIME_S
    model.Params.Threads = THREADS
    model.Params.MIPGap = MIP_GAP
    candidates = list(u.index)
    origins = list(w.index)
    existing = list(q.index)
    pairs = [(i, h) for i in origins for h in dist.columns
             if distancia_maxima is None or float(dist.loc[i, h]) <= distancia_maxima]
    allowed = set(pairs)
    if any(not any((i, h) in allowed for h in dist.columns) and w.loc[i] > 0 for i in origins):
        raise ValueError("Há setores sem nenhuma escola elegível: revise distância máxima.")
    z = model.addVars(pairs, lb=0.0, vtype=GRB.CONTINUOUS, name="fluxo")
    y = model.addVars(candidates, vtype=GRB.BINARY, name="abrir")
    for i in origins:
        model.addConstr(gp.quicksum(z[i, h] for h in dist.columns if (i, h) in allowed) == int(w.loc[i]),
                        name=f"demanda_{i}")
    for h in existing:
        model.addConstr(gp.quicksum(z[i, h] for i in origins if (i, h) in allowed) <= int(q.loc[h]),
                        name=f"capacidade_existente_{h}")
    for h in candidates:
        model.addConstr(gp.quicksum(z[i, h] for i in origins if (i, h) in allowed) <= int(u.loc[h]) * y[h],
                        name=f"capacidade_nova_{h}")
    if p is not None:
        model.addConstr(y.sum() <= int(p), name="limite_novas")
    if objetivo == "aberturas":
        model.setObjective(y.sum(), GRB.MINIMIZE)
    else:
        model.setObjective(gp.quicksum(float(dist.loc[i,h]) * z[i,h] for i,h in pairs),
                           GRB.MINIMIZE)
    return model, z, y

def atributos_solver(model):
    status = {GRB.OPTIMAL: "OPTIMAL", GRB.INFEASIBLE: "INFEASIBLE",
              GRB.TIME_LIMIT: "TIME_LIMIT", GRB.INF_OR_UNBD: "INF_OR_UNBD",
              GRB.SUBOPTIMAL: "SUBOPTIMAL"}.get(model.Status, str(model.Status))
    return {"status": status, "tem_solucao": bool(model.SolCount),
            "tempo_solver_s": float(model.Runtime), "variaveis": int(model.NumVars),
            "restricoes": int(model.NumConstrs),
            "objetivo": float(model.ObjVal) if model.SolCount else None,
            "limite_inferior": float(model.ObjBound) if model.SolCount else None,
            "gap": float(model.MIPGap) if model.SolCount else None}

## 3. Teste unitário sintético (executa antes de usar dados reais)

Dois setores com 3 e 2 estudantes; uma escola existente com capacidade 2 e uma candidata com capacidade 5. Sem nova escola, a instância não pode atender a demanda total. A solução de abertura mínima tem de abrir exatamente uma nova escola. Este teste verifica especialmente o vínculo entre \(z\) e \(y\).

In [ ]:
w_t = pd.Series([3, 2], index=["A", "B"], dtype=int)
q_t = pd.Series([2], index=["E_teste"], dtype=int)
u_t = pd.Series([5], index=["N_teste"], dtype=int)
d_t = pd.DataFrame([[1., 2.], [3., 1.]], index=w_t.index, columns=[*q_t.index, *u_t.index])
modelo_t, z_t, y_t = construir_modelo(w_t, q_t, u_t, d_t, objetivo="aberturas")
modelo_t.optimize()
assert modelo_t.Status == GRB.OPTIMAL
assert abs(modelo_t.ObjVal - 1) < 1e-6
assert y_t["N_teste"].X > 0.5
modelo_t0, _, _ = construir_modelo(w_t, q_t, u_t, d_t, p=0)
modelo_t0.optimize()
assert modelo_t0.Status == GRB.INFEASIBLE, "Sem nova unidade, a instância deveria ser inviável."
print("Teste sintético: PASSOU — abertura necessária, vínculo de capacidade e inviabilidade sem abertura")

## 4. Resolver as instâncias de Lavras

Para cada etapa e cenário: obter \(p_{min}\) **somente quando a otimalidade da primeira etapa for comprovada**; em seguida resolver a menor distância com \(p\le p_{min},p_{min}+1,p_{min}+2\). Em qualquer interrupção, registrar o status e NÃO afirmar que o número de aberturas é mínimo.

Os resultados são separados por etapa/cenário/limite de distância, sem serem confundidos com execuções anteriores. Não existe reexecução automática de casos que exijam arquivos externos ausentes.

In [ ]:
def extrair_solucao(model, z, y, w, q, u, dist):
    if not model.SolCount:
        return None, None, {}
    f = pd.DataFrame([(i, h, float(var.X), float(dist.loc[i,h]))
                      for (i,h), var in z.items() if var.X > 1e-7],
                     columns=["setor", "escola", "estudantes", "distancia_km"])
    abertas = pd.DataFrame([{"candidato": h, "abrir": int(round(var.X)), "capacidade": int(u[h])}
                            for h, var in y.items() if var.X > .5])
    assert abs(f.estudantes.sum() - w.sum()) < 1e-4
    soma = f.groupby("setor").estudantes.sum().reindex(w.index, fill_value=0)
    assert np.allclose(soma.to_numpy(), w.to_numpy(), atol=1e-5)
    uso = f.groupby("escola").estudantes.sum()
    for h in q.index:
        assert uso.get(h, 0) <= q[h] + 1e-4
    for h in u.index:
        assert uso.get(h, 0) <= u[h] * int(h in set(abertas.candidato)) + 1e-4
    erro_inteiro = float(np.max(np.abs(f.estudantes - np.round(f.estudantes)))) if len(f) else 0.
    assert erro_inteiro <= 1e-4, f"Fluxos não inteiros: {erro_inteiro}"
    student_km = float((f.estudantes * f.distancia_km).sum())
    if model.ModelName and "aberturas" not in model.ModelName:  # objetivo registrado fora desta função
        pass
    indicadores = {"aberturas": len(abertas), "distancia_total_estudante_km": student_km,
                   "distancia_media_km": student_km / float(w.sum()),
                   "maior_distancia_utilizada_km": float(f.distancia_km.max()),
                   "max_erro_integralidade_fluxo": erro_inteiro}
    return f, abertas, indicadores

registros = []
for etapa in ETAPAS_EXECUTADAS:
    for cenario_demanda, cenario_capacidade in CENARIOS:
        print("Preparando", etapa, cenario_demanda, cenario_capacidade, flush=True)
        w, q, u, D = ler_instancia(etapa, cenario_demanda, cenario_capacidade)
        nome_caso = f"{etapa}__dem_{cenario_demanda}__cap_{cenario_capacidade}__raio_{DISTANCIA_MAXIMA_KM or 'livre'}"
        pasta_caso = saida / "resultados" / nome_caso
        pasta_caso.mkdir(parents=True, exist_ok=True)
        dados_base = {"etapa": etapa, "cenario_demanda": cenario_demanda,
                      "cenario_capacidade": cenario_capacidade,
                      "distancia_maxima_km": DISTANCIA_MAXIMA_KM,
                      "demanda": int(w.sum()), "capacidade_existente": int(q.sum()),
                      "capacidade_cada_nova": int(u.iloc[0]), "setores": len(w),
                      "escolas_existentes": len(q), "candidatos": len(u)}
        model_min, z_min, y_min = construir_modelo(w, q, u, D, distancia_maxima=DISTANCIA_MAXIMA_KM,
                                                  objetivo="aberturas")
        model_min.optimize()
        inicial = {**dados_base, "fase": "min_aberturas", **atributos_solver(model_min)}
        registros.append(inicial)
        if model_min.Status != GRB.OPTIMAL:
            print("Sem prova de número mínimo de aberturas:", inicial["status"])
            continue
        p_min = int(round(model_min.ObjVal))
        print("Mínimo comprovado de novas escolas:", p_min)
        for p in range(p_min, min(p_min + VARIACOES_P, len(u)) + 1):
            model, z, y = construir_modelo(w, q, u, D, p=p,
                                          distancia_maxima=DISTANCIA_MAXIMA_KM)
            model.optimize()
            registro = {**dados_base, "fase": "distancia", "p_min": p_min, "limite_aberturas": p,
                        **atributos_solver(model)}
            if model.SolCount:
                fluxo, aberturas, medidas = extrair_solucao(model, z, y, w, q, u, D)
                registro.update(medidas)
                assert np.isclose(registro["distancia_total_estudante_km"], registro["objetivo"],
                                  rtol=1e-6, atol=1e-4)
                fluxo.to_csv(pasta_caso / f"fluxos_p{p}.csv", index=False)
                aberturas.to_csv(pasta_caso / f"novas_escolas_p{p}.csv", index=False)
            registros.append(registro)
            print(etapa, "p<=", p, registro["status"], "distancia media:", registro.get("distancia_media_km"), flush=True)
        pd.DataFrame(registros).to_csv(saida / "resultados_localizacao.csv", index=False)
pd.DataFrame(registros).to_csv(saida / "resultados_localizacao.csv", index=False)
(saida / "configuracao_gurobi.json").write_text(json.dumps(config, ensure_ascii=False, indent=2))
display(pd.DataFrame(registros))
print("Resultados:", saida / "resultados_localizacao.csv")

## 5. Critérios de interpretação

- `OPTIMAL`: solução ótima **para a instância e hipóteses adotadas**.
- `TIME_LIMIT` com incumbente: solução viável conhecida, não ótimo provado.
- `INFEASIBLE`: inviabilidade comprovada com as hipóteses; não é diagnóstico de falta real de escolas.
- Ao aumentar \(p\), a menor distância ótima não pode aumentar (mesmo conjunto de dados). O mínimo de abertura difere do mínimo de distância.
- Um raio máximo de deslocamento deve ser estudado como **cenário**, com atenção especial aos setores rurais.
- Não interpretar novos candidatos como terrenos adequados nem capacidade hipotética como vagas oficiais.

**Comparações futuras:** solução heurística versus Gurobi, sensibilidade de \(\rho\), lotação, número de turmas novas e custo/raio. Não comparar tempos sem registrar CPU, parâmetros do solver e a disponibilidade da licença.